# Download responsibly

Pyrosm downloads OpenStreetMap data from three free services: Geofabrik, BBBike, and Movisda. When you search for data with a place name, pyrosm also uses the Nominatim geocoding service. These services are run by small teams or volunteers, and everyone who uses them shares the same servers and network bandwidth. Hence, it is important to use them responsibly. In practice, this means that you download the data you need once and then reuse the file, instead of downloading it again every time you run your code.

The usage policies of the services are summarised below. The examples that follow show how to download a file once and reuse it, and how to get data for several areas without unnecessary downloads. At the end of the page, we explain how pyrosm limits downloads and geocoding requests. If you let an AI coding agent write or run pyrosm code, it should follow the [Rules for AI coding agents](ai_agents.md).

## Usage policies of the services

**Geofabrik** asked its users to download responsibly in a [blog post](https://blog.geofabrik.de/index.php/2025/09/10/download-responsibly/) in September 2025, after some clients had downloaded the same files hundreds of times per day. Geofabrik recommends that you download a file only once and keep it up to date with [pyosmium-up-to-date](https://docs.osmcode.org/pyosmium/latest/user_manual/10-Replication-Tools/), which downloads only the changes made since your last update. If you need data for the whole world, download the planet file from [planet.openstreetmap.org](https://planet.openstreetmap.org) instead of collecting it from Geofabrik's extracts. If you run automated scripts, make sure that they cannot get stuck downloading the same file again and again.

**BBBike** does not have a written download policy that we know of, but its [robots.txt](https://download.bbbike.org/robots.txt) file asks web crawlers and other robots not to download its `.pbf` files. Pyrosm downloads a BBBike extract only when you ask for data in one of BBBike's cities, and it downloads only one file at a time.

We do not know of a written download policy for **Movisda** either. When you use Movisda, keep each downloaded file and reuse it, as Geofabrik recommends.

**Nominatim** has a [usage policy](https://operations.osmfoundation.org/policies/nominatim/) for its public server. You can send at most one request per second. If you geocode many places, send the requests one after another from a single machine. Set the User-Agent header of your requests so that it identifies your application. You also need to cache the results and attribute the data to OpenStreetMap. Scripts that run longer than a day or on a schedule can send at most 4 requests per minute. Systematic queries, such as geocoding all towns of a country, are not allowed.

## Check the available extracts before downloading

Before you download anything, it is a good idea to check which extracts contain your area and how large they are. With `contains_only=True`, the `find_extracts()` function lists the extracts that contain the area, from the smallest to the largest. The function does not download any OpenStreetMap data. It may download the lists of extracts that the providers publish, which pyrosm keeps for later use. Here, we check the extracts that contain central Helsinki:

In [1]:
from pyrosm import find_extracts

# Central Helsinki, as [minx, miny, maxx, maxy] in lon/lat
extracts = find_extracts([24.93, 60.16, 24.96, 60.18], contains_only=True)
extracts[["provider", "id", "bytes"]]

,provider,id,bytes
0,BBBike,Helsinki,65082181
1,Movisda,N60E024,67184647
2,Movisda,FI-18,106000130
3,Movisda,FI,715076685
4,Geofabrik,finland,773974545
5,Movisda,N60E020-10,907812145
6,Geofabrik,europe,35333797848


As we can see, the smallest extract that contains central Helsinki is BBBike's Helsinki extract (65 MB), and the largest one is Geofabrik's Europe extract (35 GB). By default (`strategy="single"`), `get_data_by_area()` downloads the first extract in this list. You can do the same check on the command line with `pyrosm find 24.93,60.16,24.96,60.18 --contains-only` (see [Command line](command_line.ipynb)). If only some places in your area matter, such as transit stops, you can make the download smaller with the [`must_cover`](downloading_data.ipynb#cover-only-the-places-that-matter) parameter.

## Download once and reuse the file

Download the data into a folder that you keep, for example a folder inside your project. If you do not specify a `directory`, pyrosm downloads the data into a temporary folder (`<tempdir>/pyrosm`), which your operating system may empty at some point. In the following, we turn on pyrosm's log messages and call `get_data()` twice for the same file:

In [2]:
import pyrosm

pyrosm.enable_logging(progress=False)

# Downloads the file
fp = pyrosm.get_data("monaco", directory="osm_data")

# Finds the file in the folder and sends no request
fp = pyrosm.get_data("monaco", directory="osm_data")

2026-10-28 22:50:05,362 pyrosm.utils.download INFO download file=monaco-latest.osm.pbf megabytes=0.670 seconds=3.721 path=/tmp/osm_data/monaco-latest.osm.pbf


As the log shows, only the first call downloaded the file. The second call found the file in the `osm_data` folder and did not contact the server at all.

When you want newer data, you can pass `update=True`. Pyrosm then asks the server whether the file has changed since it was downloaded. It downloads the file again only if the server reports a change:

In [3]:
fp = pyrosm.get_data("monaco", directory="osm_data", update=True)

# Back to warnings only
pyrosm.enable_logging("WARNING", progress=False);

2026-10-28 22:50:05,558 pyrosm.utils.download INFO download_unchanged file=monaco-latest.osm.pbf seconds=0.189 path=/tmp/osm_data/monaco-latest.osm.pbf


Geofabrik's Monaco extract has not changed since we downloaded it a moment ago, hence pyrosm keeps the file it already has. This check works only if pyrosm downloaded the existing file itself, from the same address, and you have not modified the file since then. In addition, the server needs to send a version tag or a modification date with the file (an `ETag` or `Last-Modified` header), which Geofabrik does. Otherwise, `update=True` downloads the whole file again, which is also what happens if you delete the file to get a fresh copy. If you want to keep the data for a whole country or continent up to date, use pyosmium-up-to-date instead.

If you often read the same small area from a large file, you can also save the area into a smaller file of its own with `OSM(fp, bounding_box=bbox).to_pbf("area.osm.pbf")`.

## Download several areas

If you need data for a few areas, pass them to `get_data_by_area()` in one call instead of downloading them one by one. However, the smallest extract that contains all of the areas can be very large. For example, the only extract that contains both central Helsinki and central Tallinn is Geofabrik's Europe extract (35 GB):

In [4]:
import geopandas as gpd
from shapely.geometry import box

# Central Helsinki and central Tallinn
areas = gpd.GeoSeries(
    [box(24.93, 60.16, 24.96, 60.18), box(24.73, 59.43, 24.76, 59.445)], crs=4326
)
find_extracts(areas, contains_only=True)[["provider", "id", "bytes"]]

,provider,id,bytes
0,Geofabrik,europe,35333797848


In cases like this, you can use `strategy="smallest_total"`. With this option, `get_data_by_area()` also considers combinations of smaller extracts that together cover the areas. When such a combination is smaller in total than the single extract, pyrosm downloads the smaller extracts and merges them into one file. Pyrosm also downloads and merges a combination of smaller extracts when it does not know the size of the single extract. Finally, the merged file is cropped to the bounding box of the areas:

In [5]:
from pyrosm import get_data_by_area

extract = get_data_by_area(areas, strategy="smallest_total", directory="osm_data")
for source in extract.sources:
    print(source.provider, source.extract, f"{source.bytes / 1e6:.0f} MB")

BBBike Tallinn 24 MB
BBBike Helsinki 65 MB


Here, pyrosm downloaded the city extracts of Tallinn and Helsinki (89 MB together) instead of the 35 GB extract of Europe. Of course, if you already have files that contain your areas, you can read the areas from those files without downloading anything.

## Read many areas from one file

If you need data for many areas, download one extract that contains all of them and read each area from that file. For example, use one extract when you work with all the cities of a country or with a grid of tiles. In the following example, we use the small extract of central Helsinki that comes with pyrosm. We read three districts from the same file and count the restaurants and cafés in each of them:

In [6]:
from pyrosm import OSM, get_data

fp = get_data("helsinki_pbf")
districts = {
    "Kamppi": [24.935, 60.165, 24.942, 60.171],
    "Kluuvi": [24.942, 60.168, 24.950, 60.174],
    "Kaartinkaupunki": [24.945, 60.164, 24.953, 60.168],
}
for name, bbox in districts.items():
    osm = OSM(fp, bounding_box=bbox)
    places = osm.get_pois(custom_filter={"amenity": ["restaurant", "cafe"]})
    print(f"{name}: {len(places)} restaurants and cafés")

Kamppi: 103 restaurants and cafés
Kluuvi: 76 restaurants and cafés
Kaartinkaupunki: 44 restaurants and cafés


Use one extract even if it is larger than the separate files would be together, and do not download a separate file for each area. For example, do not use a loop to download hundreds of BBBike city extracts. In its robots.txt file, BBBike tells automated programs not to download its `.pbf` files. The download limit described below does not stop this kind of loop. Pyrosm counts the downloads from each address separately, so the limit is not reached when every round of the loop downloads from a different address. If you need data for the whole world, start from the planet file.

## Scripts that run repeatedly

Tests, continuous integration (CI), and batch jobs on a computing cluster run the same code many times, often on new machines. Pyrosm cannot see the files that have been downloaded on other machines, so it cannot reuse them. Hence, these scripts need to get their data in another way.

For tests and CI pipelines, keep a small PBF file of your own study area together with your code, for example in the `tests/data` folder of your repository, and read it in the tests instead of downloading data. Most of pyrosm's own tests do the same. They read small PBF files that come with the library, such as the file that `get_data("test_pbf")` returns. To make such a file, start with an extract that you have already downloaded, and use `to_pbf()` to cut your study area from it. Here, we cut a few blocks of Monaco from the extract that we downloaded earlier on this page:

In [7]:
from pathlib import Path

monaco = pyrosm.get_data("monaco", directory="osm_data")
small = OSM(monaco, bounding_box=[7.420, 43.733, 7.424, 43.736]).to_pbf("my_area.osm.pbf")
print(Path(small).name, f"{Path(small).stat().st_size / 1000:.0f} kB")

my_area.osm.pbf 146 kB


In your own project, write the file into a folder such as `tests/data`. A file of this size can be stored in the repository and read in every run of the pipeline. If your file is too large for that, store it once on GitHub, for example as an asset of a GitHub release, and let the pipeline fetch it from there. Either way, the pipeline does not download anything from the OpenStreetMap services.

Other scripts that run repeatedly need some care too:

- If a CI pipeline really needs to download an extract, cache the download folder between runs, for example with the `actions/cache` action of GitHub Actions. Otherwise, every run on a fresh machine downloads the data again.
- On a cluster, download the data once to a location that all jobs can read. Do this before you start the jobs, and pass the file path to each job.
- If you run the same analysis repeatedly, write it as a [recipe](recipes.ipynb). When you run the recipe again, pyrosm reuses the downloaded data and the results of the steps that have not changed.

## Geocode sparingly

Each new place name that you geocode sends a request to the Nominatim server. Pyrosm caches the results for 7 days, and during that time geocoding the same place name again does not send a new request. If you know the bounding box of your area, use it instead of the place name. When you geocode, pass a `user_agent` that identifies your application, such as `user_agent="my-study/1.0 (me@example.org)"`. Geocoding a few place names is fine. If you need to geocode many place names, run your own Nominatim server or use another provider of the Nominatim API. Pass the address of the server with the `base_url` parameter.

## How pyrosm limits downloads and geocoding requests

It is easy to write a script that sends far more requests than you intended, for example one that gets stuck in a loop and downloads the same file again and again. To catch such mistakes, pyrosm limits its downloads and geocoding requests in the following ways:

- Pyrosm keeps a log of the downloaded files in `<tempdir>/pyrosm/download-log/`. If a file has already been downloaded in full from the same address three times during the last 24 hours, pyrosm refuses to download it a fourth time and raises a `RequestRefusedError`. The error message shows you where to find the log file. If you really need to download the file again, you can delete the log file, which resets the count. An AI coding agent should show you the error message and let you decide whether to delete the log file. A check made with `update=True` does not count when the file has not changed.
- On one computer, pyrosm downloads only one file at a time from each server. This also applies when several Python processes are downloading data. If a download has to wait for another one, pyrosm writes a `download_waiting` message to the log.
- If a request fails because of a network error or a temporary problem on the server (HTTP status 408, 425, 429, or 5xx), pyrosm tries again at most twice. If the server says how long to wait, pyrosm waits that long before trying again. Otherwise, it waits 1 second before the first retry and 2 seconds before the second. If the server asks to wait longer than 60 seconds, pyrosm stops and raises an error. If the server still answers with 429 (too many requests) or 503 (service unavailable) after the retries, `get_data_by_area()` does not try any other extracts from that provider during the same call.
- Pyrosm stores the geocoding results in `<tempdir>/pyrosm/geocoding/` and reuses them for 7 days. Each process sends at most one request per second to the public Nominatim server. After one process has sent 50 requests to the public server with pyrosm's default User-Agent, pyrosm refuses to send any more and raises a `RequestRefusedError`. The error message asks you to pass your own `user_agent` or to use your own server. Answers from the cache do not count towards the 50 requests.
- Every request identifies pyrosm and its version in the User-Agent, unless you pass your own. Some AI coding agents set the environment variable `AI_AGENT` or `CLAUDECODE` when they run commands. If either environment variable is set, pyrosm also adds the word `agent` to the User-Agent.

Notice that these limits do not replace the usage policies of the services, and it is still your responsibility to follow them.